# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kubrafathima2005-kf/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Rule:** Prioritize content pages that have not been updated for a long time but still receive good search impressions. These pages may benefit from a content refresh.

**Reason codes:**
- `stale_and_visible` — Page is at least 180 days without an update and has at least 500 impressions.
- `stale_only` — Page is old but has fewer than 500 impressions.
- `visible_only` — Page has good impressions but was updated more recently.
- `neither` — Page meets neither condition.

**Action:** Review the highest-scoring pages first.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/kubrafathima2005-kf/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Signal 1: Freshness tier")
print("n =", len(df))
display(
    df.groupby("freshness_tier", dropna=False)
    ["is_declining_label"]
    .agg(n="count", decline_rate="mean")
)

print("Signal 2: Impression tier")
print("n =", len(df))
display(
    df.groupby("impression_tier", dropna=False)
    ["is_declining_label"]
    .agg(n="count", decline_rate="mean")
)

Signal 1: Freshness tier
n = 30000


,n,decline_rate
freshness_tier,,
0-30,20480,0.511377
181+,174,0.471264
31-90,175,0.588571
91-180,9171,0.611057


Signal 2: Impression tier
n = 30000


,n,decline_rate
impression_tier,,
excellent,1078,0.461967
good,7205,0.586121
low,11248,0.453947
moderate,10469,0.614672


In [2]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import os

# Signals used by the rule
stale = df["freshness_tier"] == "91-180"
visible = df["impression_tier"].isin(["moderate", "good"])

# Score
df["score"] = stale.astype(int) * 2 + visible.astype(int)

# One reason code per row
df["reason_code"] = "neither"
df.loc[visible, "reason_code"] = "visible_only"
df.loc[stale, "reason_code"] = "stale_only"
df.loc[stale & visible, "reason_code"] = "stale_and_visible"

# Action
df["action"] = "monitor"
df.loc[df["score"] >= 2, "action"] = "review_refresh"

# Rank
df = df.sort_values(
    ["score", "impressions_last_30d"],
    ascending=[False, False]
).reset_index(drop=True)

df["rank"] = range(1, len(df) + 1)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

output = df[
    ["rank", "content_id", "client_id", "score",
     "reason_code", "action"]
]

output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Queue created!")
print("Total rows:", len(output))
display(output.head(10))

Queue created!
Total rows: 30000


,rank,content_id,client_id,score,reason_code,action
0,1,content_213a0e7f700e,client_19581e27de,3,stale_and_visible,review_refresh
1,2,content_ecddc7fa6f5c,client_6208ef0f77,3,stale_and_visible,review_refresh
2,3,content_df71843dcd17,client_8527a891e2,3,stale_and_visible,review_refresh
3,4,content_b93acef8e67e,client_19581e27de,3,stale_and_visible,review_refresh
4,5,content_2ec1d5d4ba01,client_19581e27de,3,stale_and_visible,review_refresh
5,6,content_c7441728db08,client_19581e27de,3,stale_and_visible,review_refresh
6,7,content_9331e5f7eeab,client_19581e27de,3,stale_and_visible,review_refresh
7,8,content_af197dbe243a,client_6208ef0f77,3,stale_and_visible,review_refresh
8,9,content_c518779160c3,client_19581e27de,3,stale_and_visible,review_refresh
9,10,content_b2ec40b9d68c,client_6208ef0f77,3,stale_and_visible,review_refresh


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
top20 = df.head(20).copy()

top20["confidence_note"] = (
    "Medium: matches the rule signals, but refresh benefit is unproven."
)

top20["what_would_make_it_wrong"] = (
    "The page may already be performing well, recently reviewed, "
    "or not need a content refresh."
)

review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review)

,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_213a0e7f700e,review_refresh,stale_and_visible,"Medium: matches the rule signals, but refresh ...","The page may already be performing well, recen..."
1,2,content_ecddc7fa6f5c,review_refresh,stale_and_visible,"Medium: matches the rule signals, but refresh ...","The page may already be performing well, recen..."
2,3,content_df71843dcd17,review_refresh,stale_and_visible,"Medium: matches the rule signals, but refresh ...","The page may already be performing well, recen..."
3,4,content_b93acef8e67e,review_refresh,stale_and_visible,"Medium: matches the rule signals, but refresh ...","The page may already be performing well, recen..."
4,5,content_2ec1d5d4ba01,review_refresh,stale_and_visible,"Medium: matches the rule signals, but refresh ...","The page may already be performing well, recen..."
5,6,content_c7441728db08,review_refresh,stale_and_visible,"Medium: matches the rule signals, but refresh ...","The page may already be performing well, recen..."
6,7,content_9331e5f7eeab,review_refresh,stale_and_visible,"Medium: matches the rule signals, but refresh ...","The page may already be performing well, recen..."
7,8,content_af197dbe243a,review_refresh,stale_and_visible,"Medium: matches the rule signals, but refresh ...","The page may already be performing well, recen..."
8,9,content_c518779160c3,review_refresh,stale_and_visible,"Medium: matches the rule signals, but refresh ...","The page may already be performing well, recen..."
9,10,content_b2ec40b9d68c,review_refresh,stale_and_visible,"Medium: matches the rule signals, but refresh ...","The page may already be performing well, recen..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
print("WEAK PICKS")
display(
    df[df["score"] == 0][
        ["content_id", "score", "reason_code", "action"]
    ].head(10)
)

print("LEAKAGE CHECK")
forbidden = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

used_features = [
    "freshness_tier",
    "impression_tier",
    "impressions_last_30d"
]

print("Forbidden label fields used:",
      set(forbidden).intersection(used_features))
print("Future-window fields used: None")

WEAK PICKS


,content_id,score,reason_code,action
20120,content_db5989a78dd3,0,neither,monitor
20121,content_a023517539fe,0,neither,monitor
20122,content_2cb567c3c89b,0,neither,monitor
20123,content_aaef01a50def,0,neither,monitor
20124,content_8451fc6f034d,0,neither,monitor
20125,content_1a9e894be2e2,0,neither,monitor
20126,content_64373b8be1b4,0,neither,monitor
20127,content_44e481c8f55b,0,neither,monitor
20128,content_e59e3357a212,0,neither,monitor
20129,content_c84a0ab98e90,0,neither,monitor


LEAKAGE CHECK
Forbidden label fields used: set()
Future-window fields used: None


**Weak picks:** Some pages may receive a low score even though they could still benefit from a refresh. The rule may miss pages with declining clicks if they do not meet the selected freshness and impression conditions.

**Leakage check:** The scoring rule uses freshness and impression signals only. It does not use `trend_direction`, `trend_pct`, or `is_declining_label`. No future-window or label-derived fields are used in the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.